# 03 · Sızıntı kontrolü ve geçerlilik maskesi (Adım 7)

**Ayarlar:** sağ panel → *Add Input* → `melihkotman/disasterm3-turkiye`; *Accelerator: None*; *Internet: On*; *Secrets*: `GITHUB_TOKEN` **Attached**. CPU çekirdeği ne kadar çoksa o kadar hızlı (4 çekirdekte tahmini 6–20 dk).

İki iş: (1) event'ler arası coğrafi örtüşme taraması (`src.leak_check`, eşikler koddan ÖNCE yazıldı), (2) öncesi/sonrası boş bölgeler için değerlendirme geçerlilik maskesi (`src.tile_validity`).

In [ ]:
import os, subprocess, time, glob, shutil
REPO_URL = "github.com/MelihKotman/Deprem-VLM-RAG.git"
if not os.path.exists("/tmp/repo"):
    tok = None
    for i in range(6):                       # Kaggle Secrets servisi 429 verebilir: bekleyip tekrar dene
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN"); break
        except Exception as e:
            print(f"secrets denemesi {i+1}/6 başarısız ({type(e).__name__}); {60*(i+1)} sn bekleniyor"); time.sleep(60*(i+1))
    if tok:
        r = subprocess.run(["git","clone","-q",f"https://{tok}@{REPO_URL}","/tmp/repo"],capture_output=True,text=True)
        if r.returncode != 0: raise SystemExit("git clone başarısız (repo adı/token?)")
        subprocess.run(["git","-C","/tmp/repo","remote","set-url","origin",f"https://{REPO_URL}"],check=True)
    else:                                     # YEDEK: Add Input ile eklenmiş kod dataset'i (kod.zip içeriği)
        hit = glob.glob("/kaggle/input/*/src/leak_check.py") + glob.glob("/kaggle/input/*/*/src/leak_check.py")
        if not hit: raise SystemExit("Secrets 429 ve yedek kod dataset'i yok")
        shutil.copytree(os.path.dirname(os.path.dirname(hit[0])), "/tmp/repo")
os.chdir("/tmp/repo")
!pip -q install -r requirements.txt
!python -m src.config

## 0. Veri sağlam mı?

In [ ]:
!python -m src.dm3_verify

## 1. Deneme (her event'ten 8 karo; birkaç dakika)

In [ ]:
!python -m src.leak_check --out /kaggle/working/leak_deneme --limit 8

`pozitif kontrol başarı oranı` ≥ 0.80 değilse DUR, tam koşuyu çalıştırma; çıktıyı gönder.

## 2. Tam koşu (öncesi ve sonrası)

In [ ]:
!python -m src.leak_check --out /kaggle/working/leak --also-post

In [ ]:
!cat /kaggle/working/leak/ozet.txt; ls /kaggle/working/leak/gorseller | head -50

## 3. Geçerlilik maskesi

In [ ]:
!python -m src.tile_validity --out /kaggle/working/validity --save-masks
!cd /kaggle/working && zip -qr leak_sonuc.zip leak validity/ozet.csv validity/tile_validity_v1.csv && ls -la leak_sonuc.zip

Sonra: *Save Version → Save & Run All (Commit)*; *Output* sekmesinden `leak_sonuc.zip` (içinde `leak/ozet.txt`, `event_matrisi.csv`, `gorseller/`, `validity/ozet.csv`, `tile_validity_v1.csv`) indir ve sohbete yükle ya da `ozet.txt` ile `validity/ozet.csv` içeriğini yapıştır.